# The Lorentz transformation: clocks, trains and light

**Professor Nano Labs · Electromagnetism**

**Try it live in your browser:** [The Lorentz transformation lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/lorentz-transformation/)
**Watch first:** [Discovering Lorentz transformation with Albert Einstein](https://www.youtube.com/watch?v=KUuaOiH9bhM)
**Watch later:** [How the beautiful unification of electricity and magnetism begins...](https://www.youtube.com/watch?v=32kmvmioc8A) · [The light wave and particle debate, from Newton to Einstein](https://www.youtube.com/watch?v=jpu36CYydkc) · [Discovering the relativistic Dirac equation with Paul Dirac and graphene](https://www.youtube.com/watch?v=QYUyAkz7DuA)

If the speed of light is the same for every observer, an event at $(x,t)$ for one observer is at

$$\begin{pmatrix}ct'\\x'\end{pmatrix}=\begin{pmatrix}\gamma&-\gamma\beta\\-\gamma\beta&\gamma\end{pmatrix}\begin{pmatrix}ct\\x\end{pmatrix},\qquad \beta=\frac vc,\qquad \gamma=\frac1{\sqrt{1-\beta^2}}$$

for another moving at $v$. This notebook

1. builds the transformation as a matrix and checks that it keeps the speed of light and the interval fixed;
2. follows the light clock, and muons coming down through the atmosphere;
3. transforms the two lightning strikes on the train into the train's frame;
4. multiplies two boosts to show how velocities add, and why rapidities add simply.

It uses the same settings as the lab ($c=1$), and checks every number quoted in the lab's "Predict first" answers. It needs NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gamma(b): return 1 / np.sqrt(1 - b * b)
def boost(b):
    # acts on column vectors (ct, x)
    g = gamma(b)
    return np.array([[g, -g * b], [-g * b, g]])
eta = np.diag([1.0, -1.0])                 # the interval: (ct)^2 - x^2 = v^T eta v

for b in [0.1, 0.6, 0.995]:
    Lm = boost(b)
    ray = Lm @ np.array([1.0, 1.0])          # a light ray, x = ct
    print(f'beta {b}: gamma {gamma(b):.4f}; a light ray becomes (ct, x) = ({ray[0]:.4f}, {ray[1]:.4f}), still speed {ray[1] / ray[0]:.6f}; '
          f'keeps the interval: {np.allclose(Lm.T @ eta @ Lm, eta)}')
    assert np.isclose(ray[1] / ray[0], 1) and np.allclose(Lm.T @ eta @ Lm, eta)

## 1 · The light clock and the muon (Questions 1 to 3)

Mirrors $L=\tfrac12$ apart, so a tick at rest takes one unit of time. Seen from the platform, half a tick of the moving clock is a right triangle: $(c\Delta t/2)^2=L^2+(v\Delta t/2)^2$.

In [ ]:
def tick_on_platform(b, L=0.5):
    # solve the light triangle for the time of one tick: (dt/2)^2 = L^2 + (b dt/2)^2
    return 2 * L / np.sqrt(1 - b * b)

for b in [0.6, 0.8]:
    dt = tick_on_platform(b)
    print(f'beta {b}: one tick takes {dt:.4f} platform ticks; in 10 platform ticks the moving clock makes {10 / dt:.3f}; '
          f'the triangle: diagonal {gamma(b):.3f} L, across {b * gamma(b):.3f} L, up 1 L')
assert np.isclose(10 / tick_on_platform(0.6), 8) and np.isclose(10 / tick_on_platform(0.8), 6)
assert np.isclose(gamma(0.6) ** 2, 1 + (0.6 * gamma(0.6)) ** 2)

# Question 2: the rider's view is the same calculation with the roles swapped; the platform moves at -0.8
print('the rider sees the platform clock tick', round(10 / tick_on_platform(-0.8), 3), 'times while hers ticks 10')

# Question 3: muons
tau, c, H = 2.1969811e-6, 299792458.0, 15000.0
b = 0.995
r_with, r_without = gamma(b) * b * c * tau, b * c * tau
f_with, f_without = np.exp(-H / r_with), np.exp(-H / r_without)
print(f'muon at {b}c: gamma {gamma(b):.4f}; mean range {r_with:.0f} m (without time dilation {r_without:.0f} m)')
print(f'fraction reaching the ground from 15 km: {f_with:.4f} (without: {f_without:.2e}); atmosphere seen by the muon: {H / gamma(b):.0f} m')
assert round(gamma(b), 1) == 10.0 and round(r_without) == 655 and round(r_with / 1000, 2) == 6.56
assert round(f_with, 2) == 0.10 and f_without < 1e-9 and round(H / gamma(b) / 1000, 1) == 1.5

d = np.linspace(0, H, 300)
plt.figure(figsize=(6.5, 3.2))
plt.plot(d / 1000, np.exp(-d / r_with), label='with time dilation'); plt.plot(d / 1000, np.exp(-d / r_without), '--', label='without')
plt.xlabel('distance travelled (km)'); plt.ylabel('muons still alive'); plt.legend(); plt.tight_layout(); plt.show()

## 2 · The train and the lightning (Questions 4 to 6)

The train has rest length $L_0=1$. On the platform it is $L=L_0/\gamma$ long and the strikes hit its ends at $t=0$: events $(ct,x)=(0,\mp L/2)$. The train's frame moves at $\beta$; other observers at any $u$.

In [ ]:
def strikes(b, u):
    L = 1 / gamma(b)
    A, B = np.array([0.0, -L / 2]), np.array([0.0, L / 2])        # back, front: (ct, x)
    A2, B2 = boost(u) @ A, boost(u) @ B
    return dict(L=L, dt=A2[0] - B2[0], dx=B2[1] - A2[1])            # t_back - t_front, x_front - x_back

for b in [0.6, 0.8]:
    s = strikes(b, b); L = s['L']
    see_front, see_back = L / (2 * (1 + b)), L / (2 * (1 - b))      # when the passenger in the middle sees each flash
    print(f'beta {b}: train on the platform {L:.3f}; in the train frame the front is struck {s["dt"]:.3f} s first, the strikes {s["dx"]:.3f} apart; '
          f'the passenger sees the front flash at {see_front:.3f} s and the back at {see_back:.3f} s (platform time)')

s = strikes(0.6, 0.6)
assert np.isclose(s['L'], 0.8) and np.isclose(s['dt'], 0.6) and np.isclose(s['dx'], 1.0)
assert np.isclose(0.8 / (2 * 1.6), 0.25) and np.isclose(0.8 / (2 * 0.4), 1.0)
assert np.isclose(strikes(0.8, 0.8)['L'], 0.6) and np.isclose(strikes(0.8, 0.8)['dt'], 0.8)

# Question 6: the interval is the same for every observer, and observers moving the other way see the back first
us = np.linspace(-0.95, 0.95, 39)
ints = [strikes(0.6, u)['dx'] ** 2 - strikes(0.6, u)['dt'] ** 2 for u in us]
print('interval for 39 observers: min', round(min(ints), 10), 'max', round(max(ints), 10), '| platform 0.8^2 =', 0.8 ** 2, '| train 1 - 0.6^2 =', round(1 - 0.36, 10))
assert np.allclose(ints, 0.64) and min(strikes(0.6, u)['dx'] for u in us) >= 0.8 - 1e-12
print('observer at -0.6c: t_back - t_front =', round(strikes(0.6, -0.6)['dt'], 3), '(negative: the back strike first)')
assert strikes(0.6, -0.6)['dt'] < 0

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(us, [strikes(0.6, u)['dt'] for u in us]); axs[0].axhline(0, color='0.7'); axs[0].set_xlabel('observer speed u/c'); axs[0].set_ylabel('t_back - t_front')
axs[1].plot([strikes(0.6, u)['dx'] for u in us], [strikes(0.6, u)['dt'] for u in us]); axs[1].plot([0, 3], [0, 3], ':', color='0.6'); axs[1].plot([0, 3], [0, -3], ':', color='0.6')
axs[1].set_xlabel('Δx'); axs[1].set_ylabel('cΔt'); axs[1].set_title('Δx² − c²Δt² = 0.64 for everyone'); plt.tight_layout(); plt.show()

## 3 · Adding velocities (Questions 7 and 8)

A ball moving at $u'$ in the train is a boost by $u'$ on top of the train's boost by $v$. Multiplying the two matrices gives a single boost, and reading off its velocity gives the addition rule, $u=(u'+v)/(1+u'v)$. With $\beta=\tanh\phi$, the boost matrix is $\begin{pmatrix}\cosh\phi&-\sinh\phi\\-\sinh\phi&\cosh\phi\end{pmatrix}$, so rapidities add.

In [ ]:
def velocity_of(Lm):
    # a boost matrix [[g, -g b], [-g b, g]]: read off b
    return -Lm[0, 1] / Lm[0, 0]

def add(u, v): return (u + v) / (1 + u * v)

for up, v in [(0.5, 0.5), (0.9, 0.9), (0.5, 0.0001)]:
    # going to the ball's frame from the platform means boosting by v, then by u'
    u_matrix = velocity_of(boost(up) @ boost(v))
    print(f"u' = {up}, v = {v}: from the matrices {u_matrix:.6f}, from the formula {add(up, v):.6f}, common sense {up + v:.4f}; "
          f"rapidities {np.arctanh(up):.4f} + {np.arctanh(v):.4f} -> tanh = {np.tanh(np.arctanh(up) + np.arctanh(v)):.6f}")
    assert np.isclose(u_matrix, add(up, v)) and np.isclose(np.tanh(np.arctanh(up) + np.arctanh(v)), add(up, v))
assert np.isclose(add(0.5, 0.5), 0.8) and round(add(0.9, 0.9), 4) == 0.9945 and round(np.arctanh(0.5), 3) == 0.549

# Question 8: a torch on a train at 0.9c
print('light from a train at 0.9c moves at', add(1.0, 0.9), '(common sense: 1.9)')
assert add(1.0, 0.9) == 1.0 and all(np.isclose(add(1.0, v), 1.0) for v in np.linspace(-0.99, 0.99, 21))

up = np.linspace(-1, 1, 201)
plt.figure(figsize=(6.5, 3.4))
plt.plot(up, add(up, 0.5), label='relativity'); plt.plot(up, up + 0.5, '--', label='common sense')
plt.axhline(1, color='orange', ls=':'); plt.xlabel("ball in the train, u'/c"); plt.ylabel('ball on the platform, u/c'); plt.title('a train at 0.5c'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
print('notebook OK: every assertion passed')